# Example of how to simulate N-CAM imagette time series


TODO! TESTING!

In [1]:
#!/usr/bin/env Python3

import os
import warnings
import numpy as np
import matplotlib.pyplot as plt
from platosim.simulation import Simulation
from platosim.plot import *
import platosim.referenceFrames as rf

warnings.simplefilter("ignore")

In [2]:
# CONFIGURE SIMULATION

# Here we configure everythinf from python

# User defined variables
simName      = 'sim0'
dataDir      = os.getcwd()
relativePath = '../platonium/examples/fastCamPhotometry'   # Relative path from PlatoSim to input files

# Start setting up the simulation
inputfile = dataDir + '/inputfile.yaml'
sim = Simulation(simName, configurationFile=inputfile, outputDir=dataDir)

# Loading PIC targets
stars = np.loadtxt(dataDir + '/starcatalog.txt')
ra  = stars[:,0]
dec = stars[:,1]
mag = stars[:,2]

# Include star catalog
sim["ObservingParameters/StarCatalogFile"] = dataDir + '/starcatalog.txt'

# Set number of exposures
sim["ObservingParameters/NumExposures"] = 100

# Set subfield size (min 8 pixels to perform photometry)
numRowsCols = sim["SubField/NumRows"] = sim["SubField/NumColumns"] = 8

# Set the group ID
group = sim["Telescope/GroupID"] = 1
raPlatform  = sim["ObservingParameters/RApointing"]  = np.deg2rad(86.79870508)  
decPlatform = sim["ObservingParameters/DecPointing"] = np.deg2rad(-46.39594703)
azimuth = sim["Telescope/AzimuthAngle"] = sim["CameraGroups/AzimuthAngle"][g]
tilt    = sim["Telescope/TiltAngle"]    =
print()
#azimuth = np.deg2rad(sim["Telescope/AzimuthAngle"][group-1])
#tilt    = np.deg2rad(sim["Telescope/TiltAngle"][group-1])

# Set EOL conditions
sim["Telescope/TransmissionEfficiency/BOL"] = sim["Telescope/TransmissionEfficiency/EOL"]
sim["CCD/CTI/Short2013/TrapDensity/BOL"] = sim["CCD/CTI/Short2013/TrapDensity/EOL"]

# Set control parameters
sim["ControlHDF5Content/WriteSubPixelImages"] = "yes"
sim["ControlHDF5Content/WritePixelMaps"]      = "yes"
sim["ControlHDF5Content/WriteStarPositions"]  = "yes"
sim["ControlHDF5Content/WriteBiasMaps"]       = "no"
sim["ControlHDF5Content/WriteSmearingMaps"]   = "no"
sim["ControlHDF5Content/WriteThroughputMaps"] = "no"
sim["ControlHDF5Content/WriteFlatfieldMap"]   = "no"
sim["ControlHDF5Content/WriteSubPixelImages"] = "no"

SyntaxError: invalid syntax (2155988487.py, line 34)

In [ ]:
# MAKE CHECKS BEFORE RUNNING SIMULATION

pixelSize       = float(sim["CCD/PixelSize"])
focalLength     = float(sim["Camera/FocalLength/ConstantValue"]) * 1000.0  # [m] -> [mm]
focalPlaneAngle = np.deg2rad(float(sim["Camera/FocalPlaneOrientation/ConstantValue"]))
solarPanelAngle = np.deg2rad(float(sim["Platform/SolarPanelOrientation"]))

# Lastly we set the subfield around the target's coordinates

subfieldIsOnCCD = sim.setSubfieldAroundCoordinates(np.deg2rad(ra[0]), np.deg2rad(dec[0]),
                                                   numRowsCols, numRowsCols)

# SELECT SUBFIELD AND RUN SIMULATION

xFP, yFP = rf.skyToFocalPlaneCoordinates(np.deg2rad(ra[0]), np.deg2rad(dec[0]), raPlatform, decPlatform,
                                      solarPanelAngle, tilt, azimuth,
                                      focalPlaneAngle, focalLength)

distanceOA = np.rad2deg(rf.gnomonicRadialDistanceFromOpticalAxis(xFP, yFP, focalLength))

print(subfieldIsOnCCD)
print(distanceOA)

In [ ]:
# Plot star on the F-CAMs CCDs

fig, ax = plt.subplots(figsize=(15, 9))
drawCCDsInSkyMollweide(fig, raPlatform, decPlatform, solarPanelAngle, tilt, azimuth, 
                       focalPlaneAngle, focalLength, pixelSize, normal=True)
drawStarsInSkyMollweide(fig, ra, dec)

In [ ]:
# RUN SIMULATION

simFile = sim.run(removeOutputFile=True)

# Remove produced log and yaml file
output_sim = dataDir + '/' + simName
os.remove(output_sim + '.log')
os.remove(output_sim + '.yaml')
#os.remove(output_sim + '.hdf5')

In [ ]:
# Extract the CCD image and the star positions
im = simFile.getImage(0)

#Plot imagette and pixel mask
# Show image (cmap suggestions: "hot", "gnuplot2","magma", and "Spectral_r")
simFile.showImage(0, showStarPositions=True, clipPercentile=8, showMaskOfStarID=1,
                              useTitle=None, colorMap='Spectral_r', showGrid=True) 